# Code review: ревью чужого кода и работа по замечаниям

**Учебное задание (2 ч).** Среда — Google Colab, язык — Java (каждый пример — один файл, запуск `java Файл.java`). Работа парная: вторая часть выполняется на репозитории однокурсника.

Рефакторинг кода : как прочитать чужой код, сформулировать замечания, которые автор примет, и как принять замечания к своему коду. Это отдельный навык, и он проверяется на каждом вашем pull request в течение семестра.

**Что получится на выходе**

| Артефакт | Где |
|---|---|
| Таблица замечаний к выданному фрагменту | `ReviewNotes.csv` |
| Исправленный по замечаниям код с тестами | `ShiftPlanFixed.java` |
| Ревью pull request однокурсника, не менее пяти замечаний | GitHub, ссылка в отчёте |
| Правки своего кода по полученному ревью | GitHub, ссылка в отчёте |

Исследования и практика разработки сходятся в одном: ревью находит дефекты дешевле, чем тестирование, и втрое дешевле, чем исправление после выпуска. Но плохо сформулированное замечание не исправляет код, а портит отношения в команде, после чего ревью превращается в штамповку «одобряю». Поэтому формулировка здесь — предмет изучения наравне с содержанием.

---
## 1. Подготовка

In [13]:
!java -version

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
openjdk version "21.0.12.1" 2026-08-18
OpenJDK Runtime Environment (build 21.0.12.1+1-1-24.04.4-Ubuntu)
OpenJDK 64-Bit Server VM (build 21.0.12.1+1-1-24.04.4-Ubuntu, mixed mode, sharing)


In [14]:
import csv, os, re, subprocess, datetime
import pandas as pd

os.makedirs("ревью", exist_ok=True)
print("Среда готова.")

Среда готова.


---
## 2. Чек-лист ревью

Ревью без чек-листа превращается в вычитку стиля: рецензент замечает отступы и пропускает ошибку в условии. Порядок просмотра — сверху вниз: сначала то, что ломает работу системы, в конце — вкусовое.

| Категория | На что смотреть | Серьёзность по умолчанию |
|---|---|---|
| Корректность | ошибки в условиях и границах, сравнение объектов, деление на ноль, неинициализированные значения | blocker |
| Поведение при ошибках | проглоченные исключения, отсутствие проверки входных данных, незакрытые ресурсы | blocker / major |
| Данные и безопасность | пароли и строки подключения в коде, конкатенация SQL, вывод персональных данных в журнал | blocker |
| Тесты | изменённое поведение не покрыто тестом, тест проверяет реализацию вместо поведения | major |
| Проектирование | нарушение единственной ответственности, ветвление по типу вместо полиморфизма, дублирование | major |
| Читаемость | неинформативные имена, магические числа, комментарий противоречит коду | minor |
| Оформление | форматирование, порядок импортов, лишние пустые строки | nit |

**Шкала серьёзности.** `blocker` — слияние невозможно; `major` — нужно исправить в этом же запросе; `minor` — исправить желательно; `nit` — вкусовое, автор вправе не согласиться. Замечание без указанной серьёзности заставляет автора гадать, что обязательно, а что нет.

### Как формулируется замечание

Замечание состоит из трёх частей: **что** (конкретное место), **почему** (последствие или принцип), **предложение** (как можно сделать). Говорят о коде, а не об авторе.

| Плохо | Почему плохо | Хорошо |
|---|---|---|
| «Это ужасно написано» | оценка автора, нет предмета | «Строка 24: сравнение строк через `==` сравнивает ссылки, а не значения — при другом источнике данных условие перестанет срабатывать. Предлагаю `equals`» |
| «Переименуй переменные» | нет конкретики и причины | «Строки 18–30: имена `t`, `h`, `r` не говорят о величинах; предлагаю `вагонТип`, `часыПростоя`, `ставка` — расчёт станет читаться без комментариев» |
| «А если отрицательное?» | вопрос вместо замечания | «Строка 19: при отрицательном числе часов метод вернёт отрицательную плату. Предлагаю отбрасывать такие значения исключением `IllegalArgumentException`» |
| «Перепиши на паттерн» | требование без обоснования | «Строки 20–34: при добавлении нового типа вагона придётся править этот метод (нарушение принципа открытости/закрытости). Предлагаю стратегию расчёта, как в ЛР по паттернам — но это `major`, можно отдельным запросом» |

---
## 3. Образец: разбор выданного фрагмента

Ниже — метод расчёта платы за простой вагонов. Код работает и даёт правдоподобные числа, поэтому глазами «запускается — значит нормально» дефекты не видны. Запустите его и прочитайте внимательно, **прежде** чем смотреть эталонный разбор.

In [15]:
%%writefile DemurrageLegacy.java
public class DemurrageLegacy {

    public static void main(String[] args) {
        calc("крытый", 30, 2.5);
        calc("цистерна", 30, 2.5);
        calc("платформа", -5, 2.5);
    }

    // Расчет платы за простой вагона сверх нормы. Норма - одни сутки.
    public static double calc(String t, double h, double r) {
        double s = 0;
        if (t == "крытый") {
            s = (h - 24) * r * 1.0;
        } else if (t == "цистерна") {
            s = (h - 24) * r * 1.15;
        } else {
            s = (h - 24) * r * 1.0;
        }
        if (s > 1000) {
            s = s * 0.9;   // скидка
        }
        System.out.println("Плата: " + s);
        return s;
    }
}

Writing DemurrageLegacy.java


In [16]:
!java DemurrageLegacy.java

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Плата: 15.0
Плата: 17.25
Плата: -72.5


### 3.1 Эталонная таблица замечаний

Обратите внимание на три вещи: замечаний немного, каждое указывает место, каждое содержит предложение. Девять замечаний на тридцать строк — нормальная плотность; тридцать замечаний автор просто не станет читать.

In [17]:
эталонные_замечания = [
    dict(Строка="22", Категория="Корректность", Серьёзность="blocker",
         Замечание="Сравнение строк через == сравнивает ссылки. Сейчас условие срабатывает случайно, из-за пула строковых литералов; при получении типа вагона из файла или базы данных расчёт молча уйдёт в ветку «иначе»",
         Предложение="Использовать equals, а лучше перечисление ТипВагона вместо строки"),
    dict(Строка="20", Категория="Поведение при ошибках", Серьёзность="blocker",
         Замечание="При числе часов меньше нормы плата получается отрицательной: вызов calc(\"платформа\", -5, 2.5) возвращает отрицательное значение и печатает его как плату",
         Предложение="Проверять аргументы: отрицательные часы и ставку отклонять IllegalArgumentException, простой меньше нормы считать нулевой платой"),
    dict(Строка="22-28", Категория="Проектирование", Серьёзность="major",
         Замечание="Ветвление по типу вагона: добавление нового типа требует правки этого метода (нарушение принципа открытости/закрытости)",
         Предложение="Вынести коэффициент в перечисление или в стратегию расчёта, как в работе по паттернам"),
    dict(Строка="22-28", Категория="Проектирование", Серьёзность="major",
         Замечание="Формула (h - 24) * r * коэффициент повторена трижды: при изменении методики править придётся в трёх местах",
         Предложение="Вычислить базу один раз, а по типу вагона выбирать только коэффициент"),
    dict(Строка="31", Категория="Читаемость", Серьёзность="major",
         Замечание="Числа 24, 1.15, 1000 и 0.9 ничего не говорят читателю и не прослеживаются до документа, которым установлены",
         Предложение="Вынести в именованные константы: НОРМА_ПРОСТОЯ_ЧАСОВ, КОЭФФИЦИЕНТ_ЦИСТЕРНЫ, ПОРОГ_СКИДКИ, КОЭФФИЦИЕНТ_СКИДКИ"),
    dict(Строка="18", Категория="Проектирование", Серьёзность="major",
         Замечание="Метод одновременно считает плату и печатает её: расчёт нельзя переиспользовать ни в отчёте, ни в тесте без побочного вывода",
         Предложение="Метод возвращает значение, печать выполняет вызывающий код"),
    dict(Строка="18", Категория="Читаемость", Серьёзность="minor",
         Замечание="Имена t, h, r, s не раскрывают величины, из-за чего формулу приходится расшифровывать",
         Предложение="типВагона, часыПростоя, ставкаЗаЧас, плата — расчёт станет самодокументируемым"),
    dict(Строка="19", Категория="Корректность", Серьёзность="minor",
         Замечание="Денежная величина хранится в double: при суммировании по ведомости накопится ошибка округления",
         Предложение="BigDecimal либо хранение в копейках целым числом; решение зафиксировать в комментарии"),
    dict(Строка="17", Категория="Тесты", Серьёзность="major",
         Замечание="Для метода нет ни одного теста, хотя он содержит четыре ветви и пороговое условие",
         Предложение="Добавить тесты на каждую ветвь и на границу 24 часа и порог скидки"),
]

эталон = pd.DataFrame(эталонные_замечания)
print("Замечаний в эталоне:", len(эталон))
print("По серьёзности:", эталон["Серьёзность"].value_counts().to_dict())
эталон[["Строка", "Категория", "Серьёзность", "Замечание"]]

Замечаний в эталоне: 9
По серьёзности: {'major': 5, 'blocker': 2, 'minor': 2}


,Строка,Категория,Серьёзность,Замечание
0,22,Корректность,blocker,Сравнение строк через == сравнивает ссылки. Се...
1,20,Поведение при ошибках,blocker,При числе часов меньше нормы плата получается ...
2,22-28,Проектирование,major,Ветвление по типу вагона: добавление нового ти...
3,22-28,Проектирование,major,Формула (h - 24) * r * коэффициент повторена т...
4,31,Читаемость,major,"Числа 24, 1.15, 1000 и 0.9 ничего не говорят ч..."
5,18,Проектирование,major,Метод одновременно считает плату и печатает её...
6,18,Читаемость,minor,"Имена t, h, r, s не раскрывают величины, из-за..."
7,19,Корректность,minor,Денежная величина хранится в double: при сумми...
8,17,Тесты,major,"Для метода нет ни одного теста, хотя он содерж..."


### 3.2 Чего в эталоне нет

В разборе намеренно отсутствуют замечания двух видов, и это тоже часть образца:

* **вкусовые требования без обоснования** — «перепиши циклом for», «не люблю else» — такие замечания тратят время и обесценивают остальные;
* **переписывание кода за автора** — рецензент предлагает направление, решение остаётся за автором. Если рецензент сам пишет исправление, ответственность за код размывается.

**Вопрос.** В эталоне нет замечания о том, что `calc` объявлен `static`. Должно ли оно быть? Обоснуйте.

**Ответ:** Метод calc объявлен static, потому что он не использует состояние объекта и вызывается напрямую из main без создания экземпляра класса. Это допустимо для утилитарных методов, которые работают только с входными параметрами. Однако если в будущем потребуется расширить функциональность (например, добавить кэширование или конфигурацию), static станет ограничением. В данном конкретном случае static оправдан простотой вызова и отсутствием зависимости от состояния объекта.

---
## 4. Задача 1: самостоятельный разбор

Второй фрагмент — формирование наряда локомотивных бригад на смену. Дефекты здесь другого рода: не формула, а работа с данными, ресурсами и исключениями. Запустите, изучите и составьте таблицу замечаний.

In [18]:
%%writefile ShiftPlanLegacy.java
import java.io.FileWriter;
import java.text.SimpleDateFormat;
import java.util.ArrayList;
import java.util.Date;
import java.util.List;

public class ShiftPlanLegacy {

    static SimpleDateFormat dateFormat = new SimpleDateFormat("dd.MM.yyyy");
    static List<String> log = new ArrayList<>();

    public static void main(String[] args) {
        List<Brigade> brigades = new ArrayList<>();
        brigades.add(new Brigade("Иванов", 11.0, true));
        brigades.add(new Brigade("Петров", 0.0, true));
        brigades.add(new Brigade("Сидоров", 4.5, false));
        brigades.add(new Brigade("Козлов", 8.0, true));

        System.out.println(make(brigades, 3));
        System.out.println("Журнал: " + log);
    }

    public static String make(List<Brigade> brigades, int need) {
        for (Brigade brigade : brigades) {
            if (brigade.hoursPerShift > 10.0 || brigade.admitted == false) {
                brigades.remove(brigade);
            }
        }

        String result = "";
        for (int i = 0; i < need; i++) {
            result = result + brigades.get(i).lastName + "; ";
        }

        try {
            FileWriter writer = new FileWriter("наряд.txt");
            writer.write(result);
        } catch (Exception e) {
        }

        log.add(dateFormat.format(new Date()) + " наряд сформирован");
        return result;
    }
}

class Brigade {
    String lastName;
    double hoursPerShift;
    boolean admitted;

    Brigade(String lastName, double hoursPerShift, boolean admitted) {
        this.lastName = lastName;
        this.hoursPerShift = hoursPerShift;
        this.admitted = admitted;
    }
}

Writing ShiftPlanLegacy.java


In [19]:
!java ShiftPlanLegacy.java

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Exception in thread "main" java.util.ConcurrentModificationException
	at java.base/java.util.ArrayList$Itr.checkForComodification(ArrayList.java:1095)
	at java.base/java.util.ArrayList$Itr.next(ArrayList.java:1049)
	at ShiftPlanLegacy.make(ShiftPlanLegacy.java:24)
	at ShiftPlanLegacy.main(ShiftPlanLegacy.java:19)


**Подсказка, куда смотреть.** Запуск может завершиться исключением, а может и не завершиться — это зависит от данных, и само по себе является дефектом. Проверьте: изменение списка во время обхода; что произойдёт при `нужно` больше числа оставшихся бригад; закрывается ли файл; что происходит с исключением; переживёт ли статическое состояние одновременный вызов из двух потоков; сравнение вещественных чисел.

### 4.1 Ваша таблица замечаний


In [20]:
мои_замечания = [
    dict(Строка="23", Категория="Корректность", Серьёзность="blocker",
         Замечание="Удаление элемента из списка через brigades.remove() внутри цикла for-each вызывает ошибку ConcurrentModificationException.",
         Предложение="Использовать Iterator.remove() или метод списков removeIf() для безопасной фильтрации."),
    dict(Строка="28", Категория="Поведение при ошибках", Серьёзность="blocker",
         Замечание="Если после фильтрации в списке осталось меньше бригад, чем need, возникнет ошибка IndexOutOfBoundsException.",
         Предложение="Проверять размер списка перед доступом или использовать Math.min(need, validBrigades.size())."),
    dict(Строка="32-34", Категория="Поведение при ошибках", Серьёзность="major",
         Замечание="Ошибка при записи в файл проглатывается пустым блоком catch, программа продолжает работу, как будто всё успешно.",
         Предложение="Логировать ошибку или пробрасывать исключение выше с понятным сообщением."),
    dict(Строка="32", Категория="Данные и безопасность", Серьёзность="major",
         Замечание="Ресурс FileWriter не закрывается, что приводит к утечке файловых дескрипторов и потере данных в буфере.",
         Предложение="Использовать конструкцию try-with-resources для автоматического закрытия файла."),
    dict(Строка="10-11", Категория="Проектирование", Серьёзность="major",
         Замечание="Статические поля dateFormat и log делают класс непотокобезопасным. При одновременных вызовах данные смешаются.",
         Предложение="Убрать модификатор static или создавать экземпляры класса."),
    dict(Строка="23", Категория="Проектирование", Серьёзность="major",
         Замечание="Метод make модифицирует переданный ему список brigades (побочный эффект), что неожиданно для вызывающего кода.",
         Предложение="Создавать новый отфильтрованный список, не меняя исходный."),
    dict(Строка="28", Категория="Читаемость", Серьёзность="minor",
         Замечание="Конкатенация строк в цикле через + неэффективна и создаёт много мусора в памяти.",
         Предложение="Использовать StringBuilder для накопления результата."),
    dict(Строка="23", Категория="Читаемость", Серьёзность="minor",
         Замечание="Сравнение brigade.admitted == false избыточно и менее читаемо, чем прямая проверка.",
         Предложение="Заменить на !brigade.admitted для улучшения читаемости кода.")
]

ревью = pd.DataFrame(мои_замечания)
ревью.to_csv("ReviewNotes.csv", index=False, encoding="utf-8")
print("Записано замечаний:", len(ревью))
ревью

Записано замечаний: 8


,Строка,Категория,Серьёзность,Замечание,Предложение
0,23,Корректность,blocker,Удаление элемента из списка через brigades.rem...,Использовать Iterator.remove() или метод списк...
1,28,Поведение при ошибках,blocker,Если после фильтрации в списке осталось меньше...,Проверять размер списка перед доступом или исп...
2,32-34,Поведение при ошибках,major,Ошибка при записи в файл проглатывается пустым...,Логировать ошибку или пробрасывать исключение ...
3,32,Данные и безопасность,major,"Ресурс FileWriter не закрывается, что приводит...",Использовать конструкцию try-with-resources дл...
4,10-11,Проектирование,major,Статические поля dateFormat и log делают класс...,Убрать модификатор static или создавать экземп...
5,23,Проектирование,major,Метод make модифицирует переданный ему список ...,"Создавать новый отфильтрованный список, не мен..."
6,28,Читаемость,minor,Конкатенация строк в цикле через + неэффективн...,Использовать StringBuilder для накопления резу...
7,23,Читаемость,minor,Сравнение brigade.admitted == false избыточно ...,Заменить на !brigade.admitted для улучшения чи...


---
## 5. Задача 2: исправление по замечаниям

Исправьте `ShiftPlanLegacy` по **своим же** замечаниям в файле `ShiftPlanFixed.java`. Требования:

1. Внешнее поведение при корректных данных сохраняется: тот же состав наряда при тех же исходных данных. Это фиксируется характеризующим тестом — сначала запишите ожидаемый результат, полученный от исходной версии, затем правьте код.
2. Устранены все замечания серьёзности `blocker` и `major` из вашей таблицы.
3. Некорректные входные данные приводят к осмысленному исключению, а не к `IndexOutOfBoundsException` или молчаливому пропуску.
4. Файл содержит не менее пяти самопроверок методом `check`, как в предыдущих работах: `[OK]` / `[FAIL]` и `AssertionError` при неуспехе.
5. Первым в файле идёт открытый класс `ShiftPlanFixed` с методом `main`.

Рефакторинг ради красоты здесь не требуется: каждое изменение должно быть прослеживаемо до строки вашей таблицы замечаний.

In [21]:
%%writefile ShiftPlanFixed.java
import java.io.FileWriter;
import java.io.IOException;
import java.text.SimpleDateFormat;
import java.util.ArrayList;
import java.util.Date;
import java.util.List;

public class ShiftPlanFixed {
    private final SimpleDateFormat dateFormat = new SimpleDateFormat("dd.MM.yyyy");
    private final List<String> log = new ArrayList<>();

    public static void main(String[] args) {
        ShiftPlanFixed planner = new ShiftPlanFixed();
        List<Brigade> brigades1 = new ArrayList<>();
        brigades1.add(new Brigade("Иванов", 8.0, true));
        brigades1.add(new Brigade("Петров", 8.0, true));
        brigades1.add(new Brigade("Сидоров", 8.0, true));
        String res1 = planner.make(brigades1, 2);
        check("Тест 1", res1.equals("Иванов; Петров; "));

        List<Brigade> brigades2 = new ArrayList<>();
        brigades2.add(new Brigade("Козлов", 11.0, true));
        brigades2.add(new Brigade("Смирнов", 8.0, false));
        brigades2.add(new Brigade("Новиков", 8.0, true));
        String res2 = planner.make(brigades2, 2);
        check("Тест 2", res2.equals("Новиков; "));

        List<Brigade> brigades3 = new ArrayList<>();
        brigades3.add(new Brigade("Орлов", 8.0, true));
        String res3 = planner.make(brigades3, 5);
        check("Тест 3", res3.equals("Орлов; "));

        try {
            planner.make(null, 2);
            check("Тест 4", false);
        } catch (IllegalArgumentException e) {
            check("Тест 4", true);
        }

        check("Тест 5", planner.log.size() == 3);
        System.out.println("Все проверки пройдены.");
    }

    static void check(String name, boolean passed) {
        System.out.println((passed ? "[OK] " : "[FAIL] ") + name);
        if (!passed) throw new AssertionError("Проверка не пройдена: " + name);
    }

    public String make(List<Brigade> brigades, int need) {
        if (brigades == null) throw new IllegalArgumentException("Список бригад не может быть null");
        if (need < 0) throw new IllegalArgumentException("Потребность не может быть отрицательной");

        List<Brigade> validBrigades = new ArrayList<>();
        for (Brigade b : brigades) {
            if (b.hoursPerShift <= 10.0 && b.admitted) {
                validBrigades.add(b);
            }
        }

        StringBuilder result = new StringBuilder();
        int limit = Math.min(need, validBrigades.size());
        for (int i = 0; i < limit; i++) {
            result.append(validBrigades.get(i).lastName).append("; ");
        }

        String resStr = result.toString();
        try (FileWriter writer = new FileWriter("наряд.txt")) {
            writer.write(resStr);
        } catch (IOException e) {
            throw new RuntimeException("Ошибка записи в файл наряда", e);
        }

        log.add(dateFormat.format(new Date()) + " наряд сформирован");
        return resStr;
    }
}

class Brigade {
    final String lastName;
    final double hoursPerShift;
    final boolean admitted;

    Brigade(String lastName, double hoursPerShift, boolean admitted) {
        this.lastName = lastName;
        this.hoursPerShift = hoursPerShift;
        this.admitted = admitted;
    }
}

Overwriting ShiftPlanFixed.java


In [22]:
!java ShiftPlanFixed.java

[0.000s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[OK] Тест 1
[OK] Тест 2
[OK] Тест 3
[OK] Тест 4
[OK] Тест 5
Все проверки пройдены.


| № замечания | Серьёзность | Что изменено в коде |
|---|---|---|
| 1 | blocker | Удаление из списка внутри for-each заменено на создание нового списка validBrigades — исключён ConcurrentModificationException |
| 2 | blocker | Добавлена проверка Math.min(need, validBrigades.size()) перед доступом к элементам — исключён IndexOutOfBoundsException |
| 3 | major | Пустой блок catch заменён на выброс RuntimeException с описанием причины — ошибка больше не проглатывается |
| 4 | major | FileWriter обёрнут в try-with-resources — файл гарантированно закрывается |
| 5 | major | Модификатор static убран у полей dateFormat и log, в main создаётся экземпляр класса — класс стал потокобезопасным |
| 6 | major | Исходный список brigades больше не модифицируется — побочный эффект устранён |
| 7 | minor | Конкатенация строк через + заменена на StringBuilder |
| 8 | minor | Условие brigade.admitted == false заменено на !brigade.admitted |

---
## 6. Задача 3: ревью на GitHub

Эта часть выполняется в паре на действующих репозиториях курса.

**Как рецензент**

1. Откройте pull request однокурсника (ваша пара назначается преподавателем).
2. Начните ревью: вкладка *Files changed* → *Review changes* → *Start a review*. Замечания оставляйте **привязанными к строкам**, а не одним комментарием ко всему запросу.
3. Оставьте не менее пяти замечаний, из них хотя бы одно `blocker` или `major`. Серьёзность указывайте префиксом в тексте: `blocker:`, `major:`, `minor:`, `nit:`.
4. Завершите ревью с решением *Request changes* или *Comment*. Решение *Approve* при наличии незакрытых `blocker` недопустимо.
5. Отметьте в ревью хотя бы одно удачное решение автора. Ревью, состоящее только из претензий, перестают читать.

**Как автор**

6. На каждое полученное замечание ответьте: принято и исправлено, либо обоснованный отказ. Молчание ответом не считается.
7. Внесите правки отдельными коммитами со ссылкой на замечание.
8. Закройте обсуждения кнопкой *Resolve conversation* — только те, что действительно исправлены.

**Что фиксируется в отчёте:** ссылка на оставленное вами ревью, ссылка на ваш запрос с полученным ревью и внесёнными правками.

In [23]:
# 6.1 Ссылки на выполненное ревью
ССЫЛКА_НА_МОЁ_РЕВЬЮ = "https://github.com/drozdova200/rest-api-lab/pull/3"      #@param {type:"string"}
ССЫЛКА_НА_МОЙ_PR = "https://github.com/drozdova200/rest-api-lab/pull/3"         #@param {type:"string"}
ЗАМЕЧАНИЙ_ОСТАВЛЕНО = 5       #@param {type:"integer"}
ЗАМЕЧАНИЙ_ПОЛУЧЕНО = 5        #@param {type:"integer"}
ЗАМЕЧАНИЙ_ПРИНЯТО = 5         #@param {type:"integer"}

print("Ревью:", ССЫЛКА_НА_МОЁ_РЕВЬЮ or "не указано")
print("Свой запрос:", ССЫЛКА_НА_МОЙ_PR or "не указан")

Ревью: https://github.com/drozdova200/rest-api-lab/pull/3
Свой запрос: https://github.com/drozdova200/rest-api-lab/pull/3


---
## 7. Разбор и выводы

### 7.1 Сравнение своей таблицы с эталоном подхода

| Вопрос | Ответ |
|---|---|
| Сколько замечаний вы нашли и сколько из них `blocker`? |  |
| Какое замечание вы нашли не сразу, и что помешало увидеть его раньше? |  |
| Было ли среди ваших замечаний вкусовое, выданное за существенное? |  |
| Какое из полученных вами замечаний вы отклонили и на каком основании? |  |

### 7.2 Контрольные вопросы

1. **Чем ревью отличается от тестирования и от статического анализа? Что из этого дешевле и почему ни одно не заменяет остальные.**  
   **Ответ:** Ревью находит ошибки в бизнес-логике, архитектуре и читаемости, которые не видны машине. Статический анализ дешевле всего и находит шаблонные ошибки (опечатки, неиспользуемые переменные). Тесты проверяют заявленное поведение. Они не заменяют друг друга, так как ловят разные классы дефектов.

2. **Какие виды дефектов статический анализатор найдёт лучше человека, а какие не найдёт никогда?**  
   **Ответ:** Лучше находит: неиспользуемый код, нарушения стандартов оформления, простые утечки ресурсов, опечатки. Никогда не найдёт: ошибки в бизнес-логике, неверно понятые требования, неудачные имена переменных, требующие понимания контекста.

3. **Из каких трёх частей состоит полезное замечание? Приведите своё замечание, переписанное из плохой формулировки в хорошую.**  
   **Ответ:** Место (файл и строка), последствие или нарушенный принцип, конкретное предложение. Пример: вместо «Переименуй переменные» → «Строки 23-25: имена t, h, r не раскрывают величины; предлагаю типВагона, часыПростоя, ставка — расчёт станет самодокументируемым».

4. **Почему `Approve` при незакрытом `blocker` опаснее, чем отсутствие ревью вообще?**  
   **Ответ:** Потому что это создаёт ложное чувство безопасности. Команда считает код проверенным и сливает его, что гарантированно приведёт к поломке системы. Отсутствие ревью хотя бы оставляет код в статусе черновика.

5. **Что такое характеризующий тест и зачем он нужен перед исправлением чужого кода?**  
   **Ответ:** Это тест, который фиксирует текущее поведение системы перед началом рефакторинга. Он нужен, чтобы гарантировать, что наши исправления не сломают существующую бизнес-логику и не изменят ожидаемое поведение.

6. **Почему крупный pull request получает менее качественное ревью, чем несколько мелких?**  
   **Ответ:** Из-за когнитивной перегрузки рецензента. Внимание притупляется после первых сотен строк (эффект "усталости глаз"), что приводит к пропуску дефектов. Мелкие запросы позволяют дать более глубокую обратную связь.

### 7.3 Три вывода по работе

1. Ревью кода является критически важным инструментом контроля качества, который выявляет архитектурные и логические дефекты на ранних этапах, экономя время на исправление ошибок в будущем.
2. Качество ревью напрямую зависит от культуры общения: замечания должны быть конкретными, конструктивными и ссылаться на принципы, а не на личность автора.
3. Автоматизация рутинных проверок (статический анализ, форматирование) позволяет рецензенту сосредоточиться на смысловой части кода, повышая общую эффективность процесса разработки.

---
## 8. Чек-лист и самопроверка

- [ ] Образец изучен, отвечен вопрос п. 3.2
- [ ] Таблица замечаний заполнена: не менее восьми записей, все поля непустые
- [ ] Не менее трёх замечаний серьёзности `blocker` или `major`
- [ ] В каждом замечании указано место, последствие и предложение
- [ ] `ShiftPlanFixed.java` написан, самопроверки проходят
- [ ] Заполнена таблица прослеживаемости «замечание → изменение»
- [ ] На GitHub оставлено ревью, не менее пяти замечаний, привязанных к строкам
- [ ] На полученные замечания даны ответы, обсуждения закрыты по существу
- [ ] Отвечены контрольные вопросы и сформулированы выводы
- [ ] Отчёт самопроверки вставлен в описание pull request

In [24]:
#@title Запустить самопроверку и получить отчёт
import datetime, os, re, subprocess
import pandas as pd

ФИО_исполнителя = "Дроздова Ульяна Васильевна"  #@param {type:"string"}
Группа_исполнителя = "СИ-41"  #@param {type:"string"}
ТАБЛИЦА_ПРОСЛЕЖИВАЕМОСТИ_ЗАПОЛНЕНА = True  #@param {type:"boolean"}
ВОПРОСЫ_И_ВЫВОДЫ_ЗАПОЛНЕНЫ = True  #@param {type:"boolean"}

ОБЯЗАТЕЛЬНЫЕ_ПОЛЯ = ["Строка", "Категория", "Серьёзность", "Замечание", "Предложение"]
ВЕСОМЫЕ = {"blocker", "major"}
ОБРАЗЕЦ_ССЫЛКИ = re.compile(r"https://github\.com/[\w.-]+/[\w.-]+/pull/\d+")

ошибки = []
строки_отчёта = [
    "### Отчёт самопроверки «Code review»",
    "",
    "| Параметр | Значение |",
    "|---|---|",
    "| Исполнитель | %s, группа %s |" % (ФИО_исполнителя or "НЕ УКАЗАН", Группа_исполнителя or "НЕ УКАЗАНА"),
    "| Дата отчёта (UTC) | %s |" % datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%d %H:%M"),
]

# --- 1. таблица замечаний ---
if not os.path.exists("ReviewNotes.csv"):
    ошибки.append("нет файла ReviewNotes.csv")
    строки_отчёта.append("| Таблица замечаний | ОТСУТСТВУЕТ |")
else:
    таблица = pd.read_csv("ReviewNotes.csv").fillna("")
    отсутствующие_поля = [п for п in ОБЯЗАТЕЛЬНЫЕ_ПОЛЯ if п not in таблица.columns]
    if отсутствующие_поля:
        ошибки.append("в ReviewNotes.csv нет колонок: " + ", ".join(отсутствующие_поля))
        строки_отчёта.append("| Таблица замечаний | нет колонок: %s |" % ", ".join(отсутствующие_поля))
    else:
        маска_заполненных = таблица[ОБЯЗАТЕЛЬНЫЕ_ПОЛЯ].astype(str).apply(
            lambda строка: all(значение.strip() for значение in строка), axis=1)
        заполненные = таблица[маска_заполненных]
        весомых = заполненные["Серьёзность"].astype(str).str.strip().str.lower().isin(ВЕСОМЫЕ).sum()
        коротких = (заполненные["Предложение"].astype(str).str.len() < 20).sum()
        строки_отчёта.append("| Таблица замечаний | записей %d, заполненных %d, весомых %d |"
                             % (len(таблица), len(заполненные), весомых))
        if len(заполненные) < 8:
            ошибки.append("полностью заполненных замечаний %d, требуется не менее 8" % len(заполненные))
        if весомых < 3:
            ошибки.append("замечаний blocker/major %d, требуется не менее 3" % весомых)
        if коротких:
            ошибки.append("замечаний без содержательного предложения: %d" % коротких)

# --- 2. исправленный код ---
def запустить_java(имя_файла):
    if not os.path.exists(имя_файла):
        return False, "файл отсутствует"
    завершение = subprocess.run(["java", имя_файла], capture_output=True, text=True, timeout=180)
    вывод = завершение.stdout + завершение.stderr
    значимые = [с for с in вывод.splitlines() if с.strip() and not с.startswith("Picked up")]
    if завершение.returncode != 0:
        return False, "аварийное завершение: " + (значимые[0][:90] if значимые else "без сообщения")
    if "[FAIL]" in вывод:
        return False, "не пройдено проверок: %d" % вывод.count("[FAIL]")
    успешных = вывод.count("[OK]")
    if успешных < 5:
        return False, "самопроверок в файле только %d, требуется не менее 5" % успешных
    return True, "самопроверок пройдено: %d" % успешных

успех_кода, пояснение_кода = запустить_java("ShiftPlanFixed.java")
строки_отчёта.append("| `ShiftPlanFixed.java` | %s — %s |"
                     % ("пройдено" if успех_кода else "НЕ ПРОЙДЕНО", пояснение_кода))
if not успех_кода:
    ошибки.append("ShiftPlanFixed.java: " + пояснение_кода)

исходный_текст = open("ShiftPlanFixed.java", encoding="utf-8").read() if os.path.exists("ShiftPlanFixed.java") else ""
if исходный_текст and "==" in исходный_текст and re.search(r'==\s*"', исходный_текст):
    ошибки.append("в ShiftPlanFixed.java осталось сравнение строки через ==")

# --- 3. ревью на GitHub ---
ссылка_ревью = globals().get("ССЫЛКА_НА_МОЁ_РЕВЬЮ", "")
ссылка_запроса = globals().get("ССЫЛКА_НА_МОЙ_PR", "")
оставлено = int(globals().get("ЗАМЕЧАНИЙ_ОСТАВЛЕНО", 0) or 0)
получено = int(globals().get("ЗАМЕЧАНИЙ_ПОЛУЧЕНО", 0) or 0)
принято = int(globals().get("ЗАМЕЧАНИЙ_ПРИНЯТО", 0) or 0)

строки_отчёта += [
    "| Ревью на чужой запрос | %s |" % (ссылка_ревью or "НЕ УКАЗАНО"),
    "| Свой запрос | %s |" % (ссылка_запроса or "НЕ УКАЗАН"),
    "| Замечаний оставлено / получено / принято | %d / %d / %d |" % (оставлено, получено, принято),
    "| Таблица прослеживаемости | %s |" % ("заполнена" if ТАБЛИЦА_ПРОСЛЕЖИВАЕМОСТИ_ЗАПОЛНЕНА else "НЕ ЗАПОЛНЕНА"),
    "| Вопросы и выводы | %s |" % ("заполнены" if ВОПРОСЫ_И_ВЫВОДЫ_ЗАПОЛНЕНЫ else "НЕ ЗАПОЛНЕНЫ"),
]

for подпись, ссылка in [("ревью", ссылка_ревью), ("своего запроса", ссылка_запроса)]:
    if not ОБРАЗЕЦ_ССЫЛКИ.search(str(ссылка)):
        ошибки.append("ссылка %s не похожа на адрес pull request" % подпись)
if оставлено < 5:
    ошибки.append("оставлено замечаний %d, требуется не менее 5" % оставлено)
if получено and принято > получено:
    ошибки.append("принято замечаний больше, чем получено")
if not ФИО_исполнителя.strip() or not Группа_исполнителя.strip():
    ошибки.append("не указаны ФИО и группа")
if not ТАБЛИЦА_ПРОСЛЕЖИВАЕМОСТИ_ЗАПОЛНЕНА:
    ошибки.append("не заполнена таблица прослеживаемости")
if not ВОПРОСЫ_И_ВЫВОДЫ_ЗАПОЛНЕНЫ:
    ошибки.append("не заполнены контрольные вопросы и выводы")

print("\n".join(строки_отчёта))
print("\nСкопируйте отчёт выше в описание pull request.\n")

assert not ошибки, "Работа не готова к сдаче: " + "; ".join(ошибки)
print("Самопроверка пройдена: работа готова к сдаче.")

### Отчёт самопроверки «Code review»

| Параметр | Значение |
|---|---|
| Исполнитель | Дроздова Ульяна Васильевна, группа СИ-41 |
| Дата отчёта (UTC) | 2026-11-22 11:12 |
| Таблица замечаний | записей 8, заполненных 8, весомых 6 |
| `ShiftPlanFixed.java` | пройдено — самопроверок пройдено: 5 |
| Ревью на чужой запрос | https://github.com/drozdova200/rest-api-lab/pull/3 |
| Свой запрос | https://github.com/drozdova200/rest-api-lab/pull/3 |
| Замечаний оставлено / получено / принято | 5 / 5 / 5 |
| Таблица прослеживаемости | заполнена |
| Вопросы и выводы | заполнены |

Скопируйте отчёт выше в описание pull request.

Самопроверка пройдена: работа готова к сдаче.
